# Assignment 07: Optimizer Comparison (100 points)

**Unit 06: Programming PyTorch | AI 310**

Different optimizers navigate the loss landscape differently. In this assignment, you will implement SGD and Adam from scratch, compare their convergence behavior, and understand when to use each.

**Notation**:
- $\theta_t$ = parameters at step $t$
- $g_t = \nabla_\theta \mathcal{L}(\theta_t)$ = gradient at step $t$
- $\eta$ = learning rate

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np
from copy import deepcopy

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries.

---

## Part 1 (20 points, coding)

Implement **SGD with momentum** from scratch (no `torch.optim`).

Update rules:

$$v_t = \beta v_{t-1} + g_t$$
$$\theta_{t+1} = \theta_t - \eta v_t$$

Your optimizer must:
- Accept a list of parameters, learning rate, and momentum coefficient
- Implement `step()` and `zero_grad()` methods
- Maintain velocity buffers for each parameter

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MySGD:
    def __init__(self, params, lr=0.01, momentum=0.0):
        """
        Args:
            params: iterable of parameters (tensors with requires_grad=True)
            lr: learning rate
            momentum: momentum coefficient (0 = no momentum)
        """
        pass
    
    def step(self):
        """Perform one optimization step."""
        pass
    
    def zero_grad(self):
        """Zero all parameter gradients."""
        pass

In [ ]:
""" END OF THIS PART """
# Test: compare with torch.optim.SGD
torch.manual_seed(42)
w1 = torch.randn(3, 4, requires_grad=True)
w2 = w1.data.clone().requires_grad_(True)

opt_mine = MySGD([w1], lr=0.1, momentum=0.9)
opt_ref = torch.optim.SGD([w2], lr=0.1, momentum=0.9)

for _ in range(5):
    loss1 = (w1 ** 2).sum()
    loss1.backward()
    opt_mine.step()
    opt_mine.zero_grad()
    
    loss2 = (w2 ** 2).sum()
    loss2.backward()
    opt_ref.step()
    opt_ref.zero_grad()

assert torch.allclose(w1.data, w2.data, atol=1e-5), \
    f"MySGD does not match torch.optim.SGD after 5 steps"
print("Part 1 passed!")

---

## Part 2 (25 points, coding)

Implement **Adam** from scratch (no `torch.optim`).

Update rules:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MyAdam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        pass
    
    def step(self):
        pass
    
    def zero_grad(self):
        pass

In [ ]:
""" END OF THIS PART """
# Test: compare with torch.optim.Adam
torch.manual_seed(42)
w1 = torch.randn(3, 4, requires_grad=True)
w2 = w1.data.clone().requires_grad_(True)

opt_mine = MyAdam([w1], lr=0.01)
opt_ref = torch.optim.Adam([w2], lr=0.01)

for i in range(10):
    # Use same loss for both
    loss1 = (w1 ** 2).sum() + torch.sin(w1).sum()
    loss1.backward()
    opt_mine.step()
    opt_mine.zero_grad()
    
    loss2 = (w2 ** 2).sum() + torch.sin(w2).sum()
    loss2.backward()
    opt_ref.step()
    opt_ref.zero_grad()

assert torch.allclose(w1.data, w2.data, atol=1e-4), \
    f"MyAdam does not match torch.optim.Adam after 10 steps\nDiff: {(w1.data - w2.data).abs().max()}"
print("Part 2 passed!")

---

## Part 3 (20 points, coding)

**Compare optimizer trajectories on a 2D surface.**

Minimize the Beale function: $f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2$

The global minimum is at $(x, y) = (3, 0.5)$ with $f(3, 0.5) = 0$.

Starting from $(x_0, y_0) = (0, 0)$, run 500 steps with:
1. `MySGD` with lr=1e-4, momentum=0.9 → store final point as `final_sgd`
2. `MyAdam` with lr=0.01 → store final point as `final_adam`
3. Store the loss trajectories as `losses_sgd` and `losses_adam` (lists of floats)

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
def beale(xy):
    x, y = xy[0], xy[1]
    return ((1.5 - x + x * y) ** 2 +
            (2.25 - x + x * y ** 2) ** 2 +
            (2.625 - x + x * y ** 3) ** 2)

In [ ]:
### WRITE YOUR SOLUTION HERE ###

# SGD trajectory
# ...
final_sgd = ...     # tensor of shape (2,)
losses_sgd = ...    # list of 500 floats

# Adam trajectory
# ...
final_adam = ...     # tensor of shape (2,)
losses_adam = ...    # list of 500 floats

In [ ]:
""" END OF THIS PART """
assert len(losses_sgd) == 500 and len(losses_adam) == 500
assert final_sgd.shape == (2,) and final_adam.shape == (2,)

# Adam should converge much closer to (3, 0.5)
target = torch.tensor([3.0, 0.5])
dist_sgd = (final_sgd.detach() - target).norm().item()
dist_adam = (final_adam.detach() - target).norm().item()

print(f"SGD final: ({final_sgd[0].item():.3f}, {final_sgd[1].item():.3f}), dist={dist_sgd:.3f}")
print(f"Adam final: ({final_adam[0].item():.3f}, {final_adam[1].item():.3f}), dist={dist_adam:.3f}")
print(f"SGD final loss: {losses_sgd[-1]:.6f}")
print(f"Adam final loss: {losses_adam[-1]:.6f}")

assert losses_adam[-1] < losses_sgd[-1], "Adam should achieve lower loss than SGD on this problem"
print("Part 3 passed!")

---

## Part 4 (15 points, coding)

Implement a **cosine annealing learning rate schedule** from scratch.

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\frac{t\pi}{T_{\max}}\right)$$

Create a function that returns the learning rate at step $t$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def cosine_lr(t, T_max, eta_max=1e-3, eta_min=1e-6):
    """
    Compute learning rate at step t using cosine annealing.
    
    Args:
        t: current step (0-indexed)
        T_max: total number of steps
        eta_max: maximum (initial) learning rate
        eta_min: minimum (final) learning rate
    
    Returns:
        float: learning rate at step t
    """
    pass

In [ ]:
""" END OF THIS PART """
import math

# At t=0, LR should be eta_max
assert abs(cosine_lr(0, 100, 1e-3, 1e-6) - 1e-3) < 1e-8

# At t=T_max, LR should be eta_min
assert abs(cosine_lr(100, 100, 1e-3, 1e-6) - 1e-6) < 1e-8

# At t=T_max/2, LR should be midpoint
mid_lr = cosine_lr(50, 100, 1e-3, 1e-6)
expected_mid = (1e-3 + 1e-6) / 2
assert abs(mid_lr - expected_mid) < 1e-8, f"Expected {expected_mid}, got {mid_lr}"

# LR should be monotonically decreasing
lrs = [cosine_lr(t, 100) for t in range(101)]
for i in range(len(lrs) - 1):
    assert lrs[i] >= lrs[i+1], f"LR should be non-increasing: lr[{i}]={lrs[i]} > lr[{i+1}]={lrs[i+1]}"

print("Part 4 passed!")

---

## Part 5 (20 points, non-coding)

Answer the following questions about optimizer behavior.

1. Why does Adam often converge faster than SGD in the early stages of training?

2. SGD with momentum can sometimes achieve better final accuracy than Adam on image classification tasks (e.g., ResNet on ImageNet). Why might this be?

3. What is the difference between `weight_decay` in `torch.optim.Adam` vs `torch.optim.AdamW`? Why does it matter?

4. In USAAIO Round 2 (4-hour time limit), would you choose SGD or Adam? Justify your answer.

5. If your model's loss suddenly explodes to NaN during training with Adam, list 3 possible causes and fixes.

### WRITE YOUR SOLUTION HERE ###

1. *Your answer here*

2. *Your answer here*

3. *Your answer here*

4. *Your answer here*

5. *Your answer here*

""" END OF THIS PART """